# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook guides you in loading, exploring, and processing the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset package using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, referring to data elements by their Croissant `@id`s.

### Dataset Source
Croissant schema URL: <https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json>

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review and enumerate available record sets, fields, and their IDs. All navigation of the data (record sets, fields, columns) is done by their Croissant `@id`.

In [ ]:
# List all record sets in the dataset with their @id and some field @ids
record_sets = list(dataset.record_sets.keys())
print("Record sets in the dataset:")
for rs_id in record_sets:
    record_set = dataset.record_sets[rs_id]
    print(f"- Record set @id: {rs_id}")
    print(f"  Name: {record_set.name}")
    print(f"  Description: {record_set.description}")
    print(f"  Fields by @id:")
    for field in record_set.fields.values():
        print(f"    - {field.id} | name: {field.name}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# For this dataset, inspect the main data table

# Find the main record set by heuristics (choose the first if unsure)
main_record_set_id = record_sets[0]  # Assumes the main table is first. Adjust manually if needed.
print(f"Using record set @id: {main_record_set_id}")

# Load all record sets to DataFrames
dataframes = {}
for rs_id in record_sets:
    df = pd.DataFrame(list(dataset.records(record_set=rs_id)))
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records for record set {rs_id}")

print(f"Available columns in main DataFrame ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering records, normalizing a numeric field, and grouping by a key attribute. All references are made using `@id`s.

In [ ]:
# Let's select a numeric field and a grouping key by inspecting the fields in the main record set.
main_fields = list(dataset.record_sets[main_record_set_id].fields.values())
print("Fields in the main record set:")
for f in main_fields:
    print(f"  @id: {f.id} | name: {f.name} | dataType: {f.data_type}")

# Select a likely numeric field for analysis (e.g., normalized protein abundance):
# Replace the following IDs with the actual ones from the print above. For demonstration we'll use the first float/integer field.
numeric_field_id = None
group_field_id = None
for f in main_fields:
    if (f.data_type or '').lower() in ("float", "number", "integer") and numeric_field_id is None:
        numeric_field_id = f.id
    if group_field_id is None and ("protein" in (f.name or "").lower() or "description" in (f.name or "").lower()):
        group_field_id = f.id
print(f"Selected numeric field @id: {numeric_field_id}")
print(f"Selected group field @id: {group_field_id}")

# Show available columns for reference
df = dataframes[main_record_set_id]

# EDA: filter for records where the numeric field is above a threshold (arbitrarily 10), then normalize
if numeric_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:\n", filtered_df.head())
    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:\n", filtered_df[[numeric_field_id, norm_col]].head())
    # Group by key field if present
    if group_field_id is not None and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print("\nGrouped data (mean of numeric field) by group field:")
        print(grouped_df.head())
else:
    print(f"Could not find numeric field @id {numeric_field_id} in DataFrame columns.")

## 5. Visualization
Visualize the distribution of the selected numeric field and compare groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping field is available, plot boxplot per group
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        top_groups = df[group_field_id].value_counts().nlargest(5).index.tolist()
        sns.boxplot(
            x=group_field_id, y=numeric_field_id, data=df[df[group_field_id].isin(top_groups)]
        )
        plt.title(f"{numeric_field_id} by {group_field_id} (Top 5 Groups)")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
This notebook demonstrated step-by-step exploration of the FAIR^2 mass spectrometry dataset using the `mlcroissant` library. We loaded metadata, inspected record set and field structure using `@id`s, extracted data into pandas DataFrames, performed basic EDA (such as filtering, normalization, and grouping), and visualized protein abundance distributions. For further analysis, consult the dataset's Croissant schema for complete field definitions and consider integrating with domain-specific bioinformatics workflows.